# 🏭 반도체 공정 DEFECT 다중 Agent 분석 시스템
### Ollama + LangChain + LangGraph 기반 3계층 Multi-Agent 파이프라인

---

## 📐 전체 구조

```
[입력] DEFECT명 + LOT ID
        │
        ▼
┌─────────────────────────────────────────┐
│  상위 Agent : 이상분석                   │
│  중간 결과 종합 → 최종 리포트 생성       │
└────────────┬────────────────────────────┘
             │
    ┌─────────┴──────────────────────────┐
    │  중간 Agent (공정별 5종)            │
    │  PHT / DRY / CVD / SPT / WET       │
    │  하위 결과 종합 → 공정 판단         │
    └─────────┬──────────────────────────┘
              │
    ┌──────────┴─────────────────────────┐
    │  하위 Agent (분석 단계 4종)         │
    │  EQP / Chamber / Layer / MODEL     │
    │  Tool 호출 → 이상 유무 판단         │
    └──────────┬─────────────────────────┘
               │
    ┌───────────┴────────────────────────┐
    │  MCP Tools (4종)                   │
    │  TREND / MAP / FDC / 설비이력      │
    │  랜덤값 리스트에서 선택하여 반환    │
    └────────────────────────────────────┘
```


## Cell 1 · 패키지 설치

In [ ]:
# Ollama 기반 실행에 필요한 패키지 설치
# (이미 설치된 경우 건너뛰어도 됩니다)
%pip install -q langchain langgraph langchain-community langchain-ollama
print("✅ 패키지 설치 완료")


## Cell 2 · Ollama 설치 & 모델 준비 안내

> **Ollama가 없는 경우** 아래 셀을 건너뛰고 **Cell 3(Mock LLM)** 부터 실행하세요.

```bash
# macOS / Linux 터미널에서 실행
curl -fsSL https://ollama.com/install.sh | sh
ollama pull gemma3:4b   # 권장 모델 (~2.5 GB)
```

| 모델 | 크기 | 특징 |
|------|------|------|
| `gemma3:4b` | ~2.5 GB | 권장, 속도/품질 균형 |
| `llama3.2:3b` | ~2 GB | 경량 |
| `qwen2.5:7b` | ~4.7 GB | 고품질 |


## Cell 3 · Import & 공통 설정

In [13]:
import random, json, sys
from typing import Any, List, Optional, Annotated

from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, BaseMessage
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.outputs import ChatResult, ChatGeneration
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict

# ── 실행 모드 선택 ────────────────────────────────────────
# USE_OLLAMA = True  →  Ollama 실제 LLM 사용
# USE_OLLAMA = False →  Mock LLM (Ollama 없이 동작)
USE_OLLAMA  = True # False          # ← 필요에 따라 True 로 변경
OLLAMA_MODEL = "gemma3:latest"   # ← 사용할 Ollama 모델명

print(f"실행 모드 : {'🟢 Ollama ({OLLAMA_MODEL})' if USE_OLLAMA else '🟡 Mock LLM (오프라인 테스트)'}")


실행 모드 : 🟢 Ollama ({OLLAMA_MODEL})


## Cell 4 · MCP Tools 정의
> 각 Tool은 지정 리스트에서 **랜덤**으로 값을 선택해 하위 Agent 에 반환합니다.

| Tool | 이상 판정값 |
|------|------------|
| TREND | 상승, 산포증가, 헌팅, 급증 |
| MAP | 특이/군집/코너/하단 MAP |
| FDC | 이상있음 |
| 설비이력 | FDC 알람, Error, BM |


In [57]:
# ─── 리스트 정의 ────────────────────────────────────────
TREND_LIST = [
    "일정","상승","일정","일정","일정",
    "산포증가","일정","헌팅","일정","급증",
    "일정","일정","일정","일정","일정",
    "일정","일정","일정","일정","일정",
]
MAP_LIST = [
    "특이 MAP 없음","특이 MAP 있음","특이 MAP 없음","군집 MAP","특이 MAP 없음",
    "특이 MAP 없음","특이 MAP 없음","특이 MAP 없음","코너 MAP","특이 MAP 없음",
    "특이 MAP 없음","특이 MAP 없음","특이 MAP 없음","하단 MAP","특이 MAP 없음",
    "특이 MAP 없음","특이 MAP 없음","특이 MAP 없음","특이 MAP 없음","특이 MAP 없음",
]
FDC_LIST = [
    "이상없음","이상없음","이상있음","이상없음","이상없음",
    "이상없음","이상없음","이상없음","이상없음","이상없음",
    "이상없음","이상없음","이상없음","이상없음","이상없음",
    "이상있음","이상없음","이상없음","이상없음","이상없음",
]
HISTORY_LIST = [
    "변경점 없음","FDC 알람","변경점 없음","Error","변경점 없음",
    "BM","변경점 없음","변경점 없음","변경점 없음","변경점 없음",
    "변경점 없음","변경점 없음","변경점 없음","변경점 없음","변경점 없음",
    "변경점 없음","변경점 없음","변경점 없음","변경점 없음","변경점 없음",
]

ABNORMAL_TREND   = {"상승","산포증가","헌팅","급증"}
ABNORMAL_MAP     = {"특이 MAP 있음","군집 MAP","코너 MAP","하단 MAP"}
ABNORMAL_FDC     = {"이상있음"}
ABNORMAL_HIST    = {"FDC 알람","Error","BM"}

# ─── Tool 함수 ───────────────────────────────────────────
@tool
def tool_trend(lot_id: str) -> dict:
    """TREND 분석 Tool: TREND 리스트에서 랜덤 선택"""
    v = random.choice(TREND_LIST)
    return {"tool":"TREND","lot_id":lot_id,"result":v,"is_abnormal":v in ABNORMAL_TREND}

@tool
def tool_map(lot_id: str) -> dict:
    """MAP 분석 Tool: MAP 리스트에서 랜덤 선택"""
    v = random.choice(MAP_LIST)
    return {"tool":"MAP","lot_id":lot_id,"result":v,"is_abnormal":v in ABNORMAL_MAP}

@tool
def tool_fdc(lot_id: str) -> dict:
    """FDC 분석 Tool: FDC 리스트에서 랜덤 선택"""
    v = random.choice(FDC_LIST)
    return {"tool":"FDC","lot_id":lot_id,"result":v,"is_abnormal":v in ABNORMAL_FDC}

@tool
def tool_equipment_history(lot_id: str) -> dict:
    """설비이력(cTTTM) Tool: 설비이력 리스트에서 랜덤 선택"""
    v = random.choice(HISTORY_LIST)
    return {"tool":"설비이력","lot_id":lot_id,"result":v,"is_abnormal":v in ABNORMAL_HIST}

ALL_TOOLS = {
    "TREND"  : tool_trend,
    "MAP"    : tool_map,
    "FDC"    : tool_fdc,
    "설비이력": tool_equipment_history,
}

# ─── 동작 확인 ───────────────────────────────────────────
print("✅ MCP Tools 정의 완료")
for name, t in ALL_TOOLS.items():
    result = t.invoke({"lot_id": "TEST-001"})
    ab = "⚠ 이상" if result["is_abnormal"] else "✓ 정상"
    print(f"  [{name}] {result['result']:15s}  →  {ab}")


✅ MCP Tools 정의 완료
  [TREND] 일정               →  ✓ 정상
  [MAP] 특이 MAP 없음        →  ✓ 정상
  [FDC] 이상없음             →  ✓ 정상
  [설비이력] 변경점 없음           →  ✓ 정상


## Cell 5 · LLM 팩토리 (Ollama / Mock 자동 선택)

In [58]:
# ─── Mock LLM (Ollama 없이 동작하는 규칙 기반 LLM) ───────
class MockLLM(BaseChatModel):
    """
    규칙 기반 Mock LLM.
    - 15 % 확률로 이상 판정 JSON 반환
    - 실제 Ollama 없이 파이프라인 전체 흐름을 테스트할 때 사용
    """

    @property
    def _llm_type(self) -> str:
        return "mock"

    def _generate(
        self,
        messages: List[BaseMessage],
        stop: Optional[List[str]] = None,
        run_manager=None,
        **kwargs: Any,
    ) -> ChatResult:
        is_ab  = random.random() < 0.15
        pool   = ["TREND 상승 감지","FDC 알람 이력","MAP 불균일 패턴","설비 BM 이력","이상 없음","정상 범위"]
        reason = random.choice(pool) if is_ab else "이상 없음"
        payload = json.dumps({"is_abnormal": is_ab, "reason": reason}, ensure_ascii=False)
        return ChatResult(generations=[ChatGeneration(message=AIMessage(content=payload))])


# ─── LLM 팩토리 ─────────────────────────────────────────
def make_llm():
    if USE_OLLAMA:
        from langchain_ollama import ChatOllama
        return ChatOllama(model=OLLAMA_MODEL, temperature=0)
    return MockLLM()

print(f"✅ LLM 팩토리 준비 완료 ({'Ollama: ' + OLLAMA_MODEL if USE_OLLAMA else 'MockLLM'})")


✅ LLM 팩토리 준비 완료 (Ollama: gemma3:latest)


## Cell 6 · 공통 LLM 판단 헬퍼

In [59]:
def judge_with_llm(llm, agent_name: str, data_rows: list[dict]) -> dict:
    """
    tool 결과 목록을 LLM에 넘겨 이상 여부를 판단합니다.
    실패 시 규칙 기반 fallback.
    """
    lines = "\n".join(
        f"  [{r['tool']}] {r['result']} ({'이상' if r['is_abnormal'] else '정상'})"
        for r in data_rows
    )
    system = (
        f"당신은 반도체 공정 분석 전문가입니다. "
        f"'{agent_name}' 분석 데이터를 검토하고 이상 여부와 원인을 "
        "한국어로 간결하게 설명하세요. "
        '응답은 JSON 형식으로만: {"is_abnormal": true/false, "reason": "설명"}'
    )
    user = f"{agent_name} 분석 데이터:\n{lines}\n이상 여부를 판단해 주세요."
    try:
        resp = llm.invoke([SystemMessage(content=system), HumanMessage(content=user)])
        raw  = resp.content.strip()
        if "```" in raw:
            raw = raw.split("```")[1].strip()
            if raw.startswith("json"):
                raw = raw[4:].strip()
        return json.loads(raw)
    except Exception:
        any_ab  = any(r["is_abnormal"] for r in data_rows)
        reasons = [r["result"] for r in data_rows if r["is_abnormal"]]
        return {"is_abnormal": any_ab, "reason": ", ".join(reasons) or "이상 없음"}


def judge_process(llm, process_name: str, lower_results: list[dict]) -> dict:
    """하위 Agent 결과를 종합해 공정 이상 여부를 LLM으로 판단합니다."""
    lines = "\n".join(
        f"  [{r['agent']}] {'이상' if r['is_abnormal'] else '정상'} / {r['reason']}"
        for r in lower_results
    )
    system = (
        f"당신은 반도체 '{process_name}' 공정 전문가입니다. "
        "하위 분석 결과를 종합해 공정 이상 여부와 핵심 원인을 한국어로 설명하세요. "
        '응답은 JSON 형식으로만: {"is_abnormal": true/false, "reason": "설명"}'
    )
    user = f"{process_name} 공정 하위 결과:\n{lines}\n이상 여부를 판단해 주세요."
    try:
        resp = llm.invoke([SystemMessage(content=system), HumanMessage(content=user)])
        raw  = resp.content.strip()
        if "```" in raw:
            raw = raw.split("```")[1].strip()
            if raw.startswith("json"):
                raw = raw[4:].strip()
        return json.loads(raw)
    except Exception:
        any_ab  = any(r["is_abnormal"] for r in lower_results)
        reasons = [r["reason"] for r in lower_results if r["is_abnormal"]]
        return {"is_abnormal": any_ab, "reason": " / ".join(reasons) or "이상 없음"}

print("✅ 판단 헬퍼 함수 준비 완료")


✅ 판단 헬퍼 함수 준비 완료


## Cell 7 · 하위 Agent 정의
`EQP` / `Chamber` / `Layer` / `MODEL` — 각각 지정된 Tool 을 호출하여 이상 유무를 판단합니다.

| Agent | 사용 Tool |
|-------|-----------|
| EQP | TREND, FDC, 설비이력 |
| Chamber | TREND, FDC, MAP |
| Layer | TREND, MAP |
| MODEL | TREND, 설비이력 |


In [60]:
class LowerAgent:
    """하위 분석 Agent: 지정 Tool 호출 → LLM 판단"""

    def __init__(self, name: str, tool_names: list[str]):
        self.name       = name
        self.tool_names = tool_names
        self.llm        = make_llm()

    def run(self, lot_id: str) -> dict[str, Any]:
        # Tool 호출
        tool_results = [
            ALL_TOOLS[t].invoke({"lot_id": lot_id})
            for t in self.tool_names if t in ALL_TOOLS
        ]
        judgment = judge_with_llm(self.llm, self.name, tool_results)
        return {
            "agent"       : self.name,
            "lot_id"      : lot_id,
            "tool_results": tool_results,
            "is_abnormal" : judgment.get("is_abnormal", False),
            "reason"      : judgment.get("reason", ""),
        }


def make_lower_agents() -> dict[str, LowerAgent]:
    return {
        "EQP"    : LowerAgent("EQP",     ["TREND", "FDC"]),
        "Chamber": LowerAgent("Chamber", ["TREND", "MAP"]),
        "Layer"  : LowerAgent("Layer",   ["MAP",   "FDC"]),
        "MODEL"  : LowerAgent("MODEL",   ["TREND", "설비이력"]),
    }

print("✅ 하위 Agent 준비 완료: EQP / Chamber / Layer / MODEL")

# 단독 동작 확인
_lower = make_lower_agents()
r = _lower["EQP"].run("TEST-001")
print(f"  EQP 테스트: is_abnormal={r['is_abnormal']}, reason={r['reason']}")


✅ 하위 Agent 준비 완료: EQP / Chamber / Layer / MODEL
  EQP 테스트: is_abnormal=True, reason=이상있음


## Cell 8 · 중간 Agent 정의
`PHT` / `DRY` / `CVD` / `SPT` / `WET` — 하위 Agent 4종을 순서대로 실행하고 공정 수준 판단을 내립니다.


In [61]:
class MiddleAgent:
    """중간 공정 Agent: 하위 Agent 4종 실행 → 공정 판단"""

    def __init__(self, process_name: str):
        self.process_name  = process_name
        self.llm           = make_llm()
        self.lower_agents  = make_lower_agents()

    def run(self, lot_id: str) -> dict[str, Any]:
        print(f"  ┌─ [{self.process_name}] 공정 분석")
        lower_results = {}
        for name, agent in self.lower_agents.items():
            res  = agent.run(lot_id)
            lower_results[name] = res
            icon = "⚠" if res["is_abnormal"] else "✓"
            tool_str = " | ".join(f"{t['tool']}:{t['result']}" for t in res["tool_results"])
            print(f"  │  {icon} {name:8s} [{tool_str}]  →  {res['reason']}")

        judgment = judge_process(self.llm, self.process_name, list(lower_results.values()))
        icon     = "⚠ 이상" if judgment["is_abnormal"] else "✓ 정상"
        print(f"  └─ [{self.process_name}] 판정: {icon}  /  {judgment['reason']}")

        return {
            "process"      : self.process_name,
            "lot_id"       : lot_id,
            "lower_results": lower_results,
            "is_abnormal"  : judgment.get("is_abnormal", False),
            "reason"       : judgment.get("reason", ""),
        }


def make_middle_agents() -> dict[str, MiddleAgent]:
    return {name: MiddleAgent(name) for name in ["PHT","DRY","CVD","SPT","WET"]}

print("✅ 중간 Agent 준비 완료: PHT / DRY / CVD / SPT / WET")


✅ 중간 Agent 준비 완료: PHT / DRY / CVD / SPT / WET


## Cell 9 · 상위 Agent 정의 — 이상분석
중간 Agent 결과를 종합하여 DEFECT 원인을 진단하고 최종 리포트를 생성합니다.


In [62]:
# DEFECT → 관련 공정 매핑
DEFECT_PROCESS_MAP = {
    "파티클" : ["PHT","DRY","CVD"],
    "CD불량" : ["PHT","DRY","SPT"],
    "두께불량": ["CVD","WET"],
    "결함"   : ["PHT","CVD","WET"],
    "스크래치": ["WET","SPT"],
    "오염"   : ["PHT","DRY","WET"],
    "default": ["PHT","DRY","CVD","SPT","WET"],
}


class UpperAgent:
    """상위 이상분석 Agent: 중간 결과 종합 → 최종 리포트 생성"""

    def __init__(self):
        self.llm = make_llm()

    def _related_processes(self, defect_name: str) -> list[str]:
        for key, procs in DEFECT_PROCESS_MAP.items():
            if key in defect_name:
                return procs
        return DEFECT_PROCESS_MAP["default"]

    def run(self, defect_name: str, lot_id: str, middle_results: dict[str, dict]) -> str:
        related  = self._related_processes(defect_name)
        relevant = {p: v for p, v in middle_results.items() if p in related}

        # 공정별 요약
        process_lines = ["[공정별 분석 결과]"]
        for p, v in relevant.items():
            icon = "⚠ 이상" if v["is_abnormal"] else "✓ 정상"
            process_lines.append(f"  {p}: {icon} → {v['reason']}")

        # 하위 분석 상세
        detail_lines = ["\n[하위 분석 상세]"]
        for p, v in relevant.items():
            for aname, lr in v.get("lower_results", {}).items():
                icon     = "⚠" if lr["is_abnormal"] else "✓"
                tool_str = " | ".join(f"{t['tool']}:{t['result']}" for t in lr.get("tool_results",[]))
                detail_lines.append(f"  {p}/{aname} {icon} [{tool_str}] → {lr['reason']}")

        summary_text = "\n".join(process_lines + detail_lines)

        system = (
            "당신은 반도체 DEFECT 분석 전문가입니다. "
            "여러 공정의 분석 결과를 종합하여 DEFECT 근본 원인을 진단하세요. "
            "한국어로 아래 구조로 답변하세요:\n"
            "1. 요약 (2~3 문장)\n"
            "2. 이상 공정 목록 및 원인\n"
            "3. 결론 및 조치 권고"
        )
        user = (
            f"DEFECT: {defect_name}\nLOT: {lot_id}\n\n"
            f"{summary_text}\n\n최종 분석 리포트를 작성해 주세요."
        )
        try:
            resp       = self.llm.invoke([SystemMessage(content=system), HumanMessage(content=user)])
            llm_report = resp.content.strip()
        except Exception as e:
            llm_report = f"LLM 리포트 생성 실패: {e}"

        sep = "=" * 62
        report = "\n".join([
            sep,
            "📋  최종 이상분석 리포트",
            f"    DEFECT : {defect_name}   |   LOT : {lot_id}",
            f"    관련공정: {', '.join(related)}",
            sep,
            summary_text,
            "",
            "─" * 62,
            "🔍  LLM 종합 분석",
            "─" * 62,
            llm_report,
            sep,
        ])
        return report

print("✅ 상위 Agent(이상분석) 준비 완료")


✅ 상위 Agent(이상분석) 준비 완료


## Cell 10 · LangGraph State & 파이프라인 빌드

In [63]:
# ─── 공유 State 정의 ────────────────────────────────────
class AnalysisState(TypedDict):
    defect_name   : str
    lot_id        : str
    lower_results : Annotated[dict[str, Any], lambda a, b: {**a, **b}]
    middle_results: Annotated[dict[str, Any], lambda a, b: {**a, **b}]
    final_report  : str
    messages      : Annotated[list, add_messages]


# ─── LangGraph 노드 함수 ────────────────────────────────
_MIDDLE_AGENTS: dict[str, MiddleAgent] | None = None
_UPPER_AGENT:  UpperAgent | None = None

def _ensure():
    global _MIDDLE_AGENTS, _UPPER_AGENT
    if _MIDDLE_AGENTS is None: _MIDDLE_AGENTS = make_middle_agents()
    if _UPPER_AGENT   is None: _UPPER_AGENT   = UpperAgent()


def node_lower(state: AnalysisState) -> dict:
    """하위 분석 노드: 모든 공정의 하위 Agent 실행"""
    _ensure()
    lot_id = state["lot_id"]
    print(f"\n{'━'*62}\n[노드 1/3] 하위 분석  LOT={lot_id}\n{'━'*62}")
    lower_results = {}
    for pname, mid in _MIDDLE_AGENTS.items():
        process_lower = {}
        for aname, lower in mid.lower_agents.items():
            process_lower[aname] = lower.run(lot_id)
        lower_results[pname] = process_lower
    return {"lower_results": lower_results,
            "messages": [AIMessage(content=f"하위 분석 완료: {lot_id}")]}


def node_middle(state: AnalysisState) -> dict:
    """중간 분석 노드: 하위 결과를 공정별로 종합 판단"""
    _ensure()
    lot_id        = state["lot_id"]
    lower_results = state.get("lower_results", {})
    print(f"\n{'━'*62}\n[노드 2/3] 중간 분석  LOT={lot_id}\n{'━'*62}")
    middle_results = {}
    for pname, mid in _MIDDLE_AGENTS.items():
        pl  = lower_results.get(pname, {})
        jud = judge_process(mid.llm, pname, list(pl.values()))
        middle_results[pname] = {
            "process"      : pname,
            "lot_id"       : lot_id,
            "lower_results": pl,
            "is_abnormal"  : jud.get("is_abnormal", False),
            "reason"       : jud.get("reason", ""),
        }
        icon = "⚠ 이상" if middle_results[pname]["is_abnormal"] else "✓ 정상"
        print(f"  [{pname}] {icon}  /  {middle_results[pname]['reason']}")
    return {"middle_results": middle_results,
            "messages": [AIMessage(content=f"중간 분석 완료: {lot_id}")]}


def node_upper(state: AnalysisState) -> dict:
    """상위 분석 노드: 최종 이상분석 리포트 생성"""
    _ensure()
    print(f"\n{'━'*62}\n[노드 3/3] 상위 이상분석\n{'━'*62}")
    report = _UPPER_AGENT.run(state["defect_name"], state["lot_id"], state["middle_results"])
    return {"final_report": report,
            "messages": [AIMessage(content="최종 리포트 생성 완료")]}


# ─── 그래프 빌드 ────────────────────────────────────────
def build_graph():
    g = StateGraph(AnalysisState)
    g.add_node("lower_analysis",  node_lower)
    g.add_node("middle_analysis", node_middle)
    g.add_node("upper_analysis",  node_upper)
    g.add_edge(START,             "lower_analysis")
    g.add_edge("lower_analysis",  "middle_analysis")
    g.add_edge("middle_analysis", "upper_analysis")
    g.add_edge("upper_analysis",  END)
    return g.compile()

GRAPH = build_graph()
print("✅ LangGraph 파이프라인 빌드 완료")
print("\n그래프 구조:")
print("  START → lower_analysis → middle_analysis → upper_analysis → END")


✅ LangGraph 파이프라인 빌드 완료

그래프 구조:
  START → lower_analysis → middle_analysis → upper_analysis → END


## Cell 11 · 그래프 구조 시각화 (선택)

In [64]:
# mermaid 다이어그램 출력 (IPython 환경)
try:
    from IPython.display import Image, display
    img = GRAPH.get_graph().draw_mermaid_png()
    display(Image(img))
except Exception:
    # 텍스트 fallback
    print(GRAPH.get_graph().draw_mermaid())


---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	lower_analysis(lower_analysis)
	middle_analysis(middle_analysis)
	upper_analysis(upper_analysis)
	__end__([<p>__end__</p>]):::last
	__start__ --> lower_analysis;
	lower_analysis --> middle_analysis;
	middle_analysis --> upper_analysis;
	upper_analysis --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## Cell 12 · 분석 실행 함수

In [65]:
def run_analysis(defect_name: str, lot_id: str) -> str:
    """분석 파이프라인 실행"""
    # Agent 캐시 초기화 (모드 전환 시 필요)
    global _MIDDLE_AGENTS, _UPPER_AGENT
    _MIDDLE_AGENTS = None
    _UPPER_AGENT   = None

    initial: AnalysisState = {
        "defect_name"  : defect_name,
        "lot_id"       : lot_id,
        "lower_results": {},
        "middle_results": {},
        "final_report" : "",
        "messages"     : [],
    }
    final = GRAPH.invoke(initial)
    return final["final_report"]

print("✅ run_analysis() 함수 준비 완료")
print()
print("사용법:")
print('  report = run_analysis("파티클", "LOT-001")')
print('  print(report)')


✅ run_analysis() 함수 준비 완료

사용법:
  report = run_analysis("파티클", "LOT-001")
  print(report)


## Cell 13 · 🚀 테스트 실행
아래 `DEFECT_NAME` / `LOT_ID` 값을 바꾸어 다양한 케이스를 테스트해 보세요.

지원 DEFECT 키워드: `파티클` / `CD불량` / `두께불량` / `결함` / `스크래치` / `오염`


In [67]:
# ─── 실행 파라미터 ──────────────────────────────────────
DEFECT_NAME = "파티클"    # 분석할 DEFECT 명
LOT_ID      = "LOT-A001"  # 대상 LOT ID

# ─── 실행 ────────────────────────────────────────────────
print("=" * 62)
print(f"🏭  반도체 공정 DEFECT 다중 Agent 분석 시스템")
print(f"    DEFECT  : {DEFECT_NAME}")
print(f"    LOT ID  : {LOT_ID}")
print(f"    LLM 모드: {'Ollama (' + OLLAMA_MODEL + ')' if USE_OLLAMA else 'Mock LLM'}")
print("=" * 62)

report = run_analysis(DEFECT_NAME, LOT_ID)
print(report)


🏭  반도체 공정 DEFECT 다중 Agent 분석 시스템
    DEFECT  : 파티클
    LOT ID  : LOT-A001
    LLM 모드: Ollama (gemma3:latest)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[노드 1/3] 하위 분석  LOT=LOT-A001
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[노드 2/3] 중간 분석  LOT=LOT-A001
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  [PHT] ⚠ 이상  /  헌팅 / 상승, 특이 MAP 있음 / 산포증가
  [DRY] ⚠ 이상  /  상승
  [CVD] ⚠ 이상  /  급증, 코너 MAP / 급증, BM
  [SPT] ⚠ 이상  /  상승
  [WET] ✓ 정상  /  이상 없음

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[노드 3/3] 상위 이상분석
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📋  최종 이상분석 리포트
    DEFECT : 파티클   |   LOT : LOT-A001
    관련공정: PHT, DRY, CVD
[공정별 분석 결과]
  PHT: ⚠ 이상 → 헌팅 / 상승, 특이 MAP 있음 / 산포증가
  DRY: ⚠ 이상 → 상승
  CVD: ⚠ 이상 → 급증, 코너 MAP / 급증, BM

[하위 분석 상세]
  PHT/EQP ⚠ [TREND:헌팅 | FDC:이상없음] → 헌팅
  PHT/Chamber ⚠ [TREND:상승 | MAP:특이 MAP 있음] → 상승, 특이 MAP 있음
  

## Cell 14 · 배치 테스트 (여러 케이스 한번에)

In [68]:
TEST_CASES = [
    ("파티클",   "LOT-A001"),
    ("CD불량",   "LOT-B042"),
    ("두께불량", "LOT-C099"),
]

reports = {}
for defect, lot in TEST_CASES:
    print(f"\n{'🔬 ' + '='*58}")
    print(f"  DEFECT={defect}  /  LOT={lot}")
    r = run_analysis(defect, lot)
    reports[f"{defect}_{lot}"] = r
    print("  ✅ 완료")

print(f"\n\n총 {len(reports)} 케이스 분석 완료")



🔬 ==========================================================
  DEFECT=파티클  /  LOT=LOT-A001

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[노드 1/3] 하위 분석  LOT=LOT-A001
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[노드 2/3] 중간 분석  LOT=LOT-A001
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  [PHT] ⚠ 이상  /  특이 MAP 있음 / 급증
  [DRY] ⚠ 이상  /  상승, 군집 MAP / 코너 MAP
  [CVD] ⚠ 이상  /  특이 MAP 있음
  [SPT] ⚠ 이상  /  상승 / Error
  [WET] ⚠ 이상  /  코너 MAP / 군집 MAP

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[노드 3/3] 상위 이상분석
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ✅ 완료

🔬 ==========================================================
  DEFECT=CD불량  /  LOT=LOT-B042

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[노드 1/3] 하위 분석  LOT=LOT-B042
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━